In [5]:
import gc
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer


In [6]:
TRAIN_IN = "/kaggle/input/datasets/zerotoinfinitya/train-preprocessed/train_preprocessed.parquet"
VAL_IN = "/kaggle/input/datasets/zerotoinfinitya/validation-preprocessed/validation_preprocessed.parquet"
TEST_IN = "/kaggle/input/datasets/zerotoinfinitya/test-preprocessed/test_preprocessed.parquet"

TRAIN_OUT = "/kaggle/working/train_source_code_features.npz"
VAL_OUT = "/kaggle/working/validation_source_code_features.npz"
TEST_OUT = "/kaggle/working/test_source_code_features.npz"
VECTORIZER_OUT = "/kaggle/working/vectorizer_source_code_20k.joblib"

In [7]:
CODE_TOKEN = r"[A-Za-z_][A-Za-z0-9_]+|[^\sA-Za-z0-9_]"

In [8]:
def read_source(path):
    df = pd.read_parquet(path, columns=["source_capped"])
    return ["" if t is None else t for t in df["source_capped"]]

In [9]:
def read_source(path):
    df = pd.read_parquet(path, columns=["source_capped"])
    return ["" if t is None else t for t in df["source_capped"]]

In [10]:
def save_sparse(path, matrix, vocab):
    np.savez_compressed(path,
                        word_data=matrix.data,
                        word_indices=matrix.indices,
                        word_indptr=matrix.indptr,
                        word_shape=np.asarray(matrix.shape),
                        vocabulary=vocab)

In [13]:
probe = TfidfVectorizer(token_pattern=CODE_TOKEN, lowercase=False)
probe.fit(["msg.sender.call.value(x)(); owner = Owner;"])

TfidfVectorizer(lowercase=False,
                token_pattern='[A-Za-z_][A-Za-z0-9_]+|[^\\sA-Za-z0-9_]')

In [14]:
vectorizer = TfidfVectorizer(
    max_features=20000,      
    ngram_range=(1, 2),       
    min_df=5,                 
    token_pattern=CODE_TOKEN,
    lowercase=False,         
    sublinear_tf=True,       
)

train_text = read_source(TRAIN_IN)
vectorizer.fit(train_text)

TfidfVectorizer(lowercase=False, max_features=20000, min_df=5,
                ngram_range=(1, 2), sublinear_tf=True,
                token_pattern='[A-Za-z_][A-Za-z0-9_]+|[^\\sA-Za-z0-9_]')

In [15]:
vocab = np.asarray([str(t) for t in vectorizer.get_feature_names_out()],
                   dtype=np.str_)
print("vocabulary:", len(vocab), "terms")


vocabulary: 20000 terms


In [16]:
train_word = vectorizer.transform(train_text).astype(np.float32).tocsr()
example_source = train_text[0]        # kept for the example at the end
del train_text
gc.collect()

30

In [19]:
val_text = read_source(VAL_IN)
val_word = vectorizer.transform(val_text).astype(np.float32).tocsr()
del val_text
gc.collect()

12

In [20]:

test_text = read_source(TEST_IN)
test_word = vectorizer.transform(test_text).astype(np.float32).tocsr()
del test_text
gc.collect()


0

In [21]:
print("train      :", train_word.shape)
print("validation :", val_word.shape)
print("test       :", test_word.shape)

train      : (79641, 20000)
validation : (10861, 20000)
test       : (15972, 20000)


In [22]:

save_sparse(TRAIN_OUT, train_word, vocab)
save_sparse(VAL_OUT, val_word, vocab)
save_sparse(TEST_OUT, test_word, vocab)
joblib.dump(vectorizer, VECTORIZER_OUT, compress=3)

print("Saved:")
print("  ", TRAIN_OUT)
print("  ", VAL_OUT)
print("  ", TEST_OUT)
print("  ", VECTORIZER_OUT)


Saved:
   /kaggle/working/train_source_code_features.npz
   /kaggle/working/validation_source_code_features.npz
   /kaggle/working/test_source_code_features.npz
   /kaggle/working/vectorizer_source_code_20k.joblib


In [23]:


# =====================================================================
# 6. ONE EXAMPLE: BEFORE AND AFTER
# =====================================================================
print()
print("=" * 60)
print("BEFORE - train contract 0, first 300 characters")
print("=" * 60)
print(example_source[:300])

row = train_word[0]
top = np.argsort(-row.data)[:15]

print()
print("=" * 60)
print("AFTER - %d of 20,000 columns are non-zero; the 15 strongest:"
      % row.nnz)
print("=" * 60)
table = pd.DataFrame({
    "term": [vocab[row.indices[i]] for i in top],
    "tf-idf": row.data[top].round(3),
})
print(table.to_string(index=False))



BEFORE - train contract 0, first 300 characters
pragma solidity 0.5.4;

interface IERC20 {





    function balanceOf(address account) external view returns (uint256);


    function transfer(address recipient, uint256 amount) external returns (bool);


    function allowance(address owner, address spender) external view returns (uint256);


   

AFTER - 824 of 20,000 columns are non-zero; the 15 strongest:
           term  tf-idf
  frozenAccount   0.140
frozenAccount [   0.133
! frozenAccount   0.128
    currentTime   0.114
         minter   0.107
        _target   0.103
    ( _balances   0.093
     ( decimals   0.093
           _who   0.092
      * uint256   0.092
      ( _target   0.085
     decimals )   0.084
          month   0.082
    FrozenFunds   0.081
  FrozenFunds (   0.081
